# Setup

In [1]:
# This makes text wrap in the output box
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))
get_ipython().events.register('pre_run_cell', set_css)

In [2]:
!git clone https://github.com/probcomp/hfppl.git
!cd hfppl && pip3 install .

Cloning into 'hfppl'...
remote: Enumerating objects: 314, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 314 (delta 47), reused 95 (delta 42), pack-reused 199
Receiving objects: 100% (314/314), 710.80 KiB | 8.46 MiB/s, done.
Resolving deltas: 100% (158/158), done.
Processing /content/hfppl
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 24.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.6/92.6 MB 9.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.4/261.4 kB 27.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 73.8 MB/s eta 0:00:00
  Created wheel for hfppl: filename=hfppl-0.1.0-py3-none-any.whl size=17811 sha256=b55d9c2dde5d372f506af523b0da0391a18ab2a9e190ac89e1ec8762e0fb5bcb
  Stored in directory: /tmp/pip-ephem-wheel-cache-

In [3]:
# Run this cell to mount your Google Drive.
# ONLY ON GOOGLE COLAB!
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/My Drive/NLP_Research_Project/"

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1ivQeIXlw8y3NWetNclDHJgki5NM272Du/NLP_Research_Project


In [22]:
from hfppl import Model, LMContext, TokenCategorical, CachedCausalLM, smc_steer, smc_standard
from score import compute_pbf_score, compute_pbi_score
import csv
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM,AutoModelForSequenceClassification, pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import asyncio

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

In [5]:
#Preprocessing the CSV file that contains the BASIL database.
# df = pd.read_csv('processed_data.csv')
df = pd.read_csv('processed_data_combined.csv')

paragraphs = df["body"] # Gets paragraphs from CSV

#This line was intended to convert stances to integers. Is not needed anymore since now we use MultiLabel Binarizer for
# multilabel classification
# df['stance'] = df['stance'].replace([ 'left', 'center', 'liberal', 'conservative', 'right'],[0,1,2,3,4])

stances = df['stance'] # Gets all the stances/labels
# Stances: {'left', 'center', 'liberal', 'conservative', 'right'}
df['body'] =df['body'].astype(str)

Load the bias prediction model

In [6]:
bias_model_name = '/content/drive/MyDrive/NLP_Research_Project/Saved_Models/politics_best_2500/politics_best_2500_30bz_000007_best'
bias_tokenizer_name = '/content/drive/MyDrive/NLP_Research_Project/Saved_Models/politics_best_2500/tokenizer_politics_best_2500_30bz_000007_best'
batch_size = 16
classes = np.array(['center', 'left', 'right'])
class2id = {
    'center': 0,
    'left': 1,
    'right': 2
}

In [7]:
bias_prediction_tokenizer = AutoTokenizer.from_pretrained(bias_tokenizer_name)

bias_prediction_model = AutoModelForSequenceClassification.from_pretrained(bias_model_name)
_ = bias_prediction_model.to(device)

In [63]:
def bias_model(text):
  text_enc = bias_prediction_tokenizer([text], truncation=True, padding=True, return_tensors='pt')

  outputs = bias_prediction_model(text_enc.input_ids.to(device), attention_mask=text_enc.attention_mask.to(device))
  logits = outputs.logits.detach().cpu()

  # Softmax makes more sense for single classifications
  sf_pred = outputs.logits.softmax(dim=-1).tolist()

  # threshold = 0.5

  # predictions = (torch.tensor(sf_pred) > threshold).int().numpy()
  pred = classes[torch.tensor(sf_pred).numpy().argmax()]
  return pred, logits[0].numpy()
  # pred = mlb.inverse_transform(predictions)[0]
  # if len(pred) != 1:
  #   print(text)
  #   print(logits)
  #   print(sf_pred)
  #   print(predictions)
  #   print(pred)
  #   return 'center'
  # return pred[0]

In [64]:
bias_model('text text text')

('left', array([-0.8390784 , -0.03293832, -2.3434815 ], dtype=float32))

# Naive SMC Steer
At each step, generate a sentence and condition that the summary so far is the target bias. This is very inefficient.

In [128]:
class NaiveModel(Model):
    def __init__(self, lm, bias_model, prompt, target_bias, max_len=512):
      super().__init__()

      lm.cache_kv(lm.tokenizer.encode(prompt))

      self.context = LMContext(lm, prompt)

      self.prompt_len = len(str(self.context.s))

      self.target_bias = target_bias

      self.max_len = max_len

      self.bias_model = bias_model

    async def gen_sentence(self):
      token = await self.sample(self.context.next_token())
      self.max_len -= 1

      while True:
        yield token
        if str(token) in ['.', '!', '?'] or token.token_id == self.context.lm.tokenizer.eos_token_id or self.max_len <= 0:
          break
        token = await self.sample(self.context.next_token())
        self.max_len -= 1

    def condition_on_bias(self):
      summary = str(self.context.s)[self.prompt_len+1:]
      bias, _ = self.bias_model(summary)
      print(bias)
      self.condition(bias == self.target_bias)

    async def step(self):
      sentence = []
      async for token in self.gen_sentence():
        sentence.append(token)

      sentence_str = llm.tokenizer.decode([t.token_id for t in sentence], skip_special_tokens=True)
      print('next sentence:', sentence_str)

      self.condition_on_bias()

      if sentence[-1].token_id == self.context.lm.tokenizer.eos_token_id or self.max_len <= 0:
        self.finish()

    def immutable_properties(self):
       return set(['target_bias', 'prompt_len', 'bias_model'])


# SMC Steer with Twist
At each step, generate a sentence and use `twist` to guide the particle on whether it's on the right track via the prediction probability of the bias. The intuition is if it has high probability of being the target bias, the summary is on the right track. Essentially incorporating the bias prediction probability into the weight of the particle.

In [129]:
# everything else the same as the Naive Model
class TwistModel(NaiveModel):
    def condition_on_bias(self):
      summary = str(self.context.s)[self.prompt_len+1:]
      _, bias_logits = self.bias_model(summary)
      print(bias_logits[class2id[self.target_bias]])
      self.twist(bias_logits[class2id[self.target_bias]])


# Generate Summary

In [15]:
model_name = "gpt2"
llm = CachedCausalLM.from_pretrained(model_name)
llm.batch_size = 8

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

You are loading your model in 8bit or 4bit but no linear modules were found in your model. this can happen for some architectures such as gpt2 that uses Conv1D instead of Linear layers. Please double check your model architecture, or submit an issue on github if you think this is a bug.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [16]:
article = df.loc[0]['body']
print(df.loc[0]['stance'])
print(article)

center
Call it Cheney versus Cheney.
Mary Cheney, one of ex-Vice President Dick Cheney’s two daughters, has taken to Facebook to blast her older sibling, Elizabeth, a Wyoming Senate candidate, for the latter’s stance on same-sex marriage, The New York Times is reporting.
Mary Cheney, openly lesbian and married to Heather Poe since 2012, reportedly posted to her personal page on the social media site: “For the record, I love my sister, but she is dead wrong on the issue of marriage.
“Freedom means freedom for everyone. That means that all families — regardless of how they look or how they are made — all families are entitled to the same rights, privileges and protections as every other.”
The Times reports Liz Cheney on Friday first articulated her position on the controversial subject, saying it should be something for voters to decide on a state-by-state basis, and not a matter for “judges” or “legislators.”
“I am not pro-gay marriage,” Liz Cheney reportedly said. “I believe the issue 

In [124]:
# Test generating a summary

prompt = f'Summarize this article: {article}'

# generator = pipeline('text-generation', model=model_name, max_length=512, return_full_text=False)
# generator(prompt)

inputs = llm.tokenizer(prompt, return_tensors="pt").input_ids.to(device)
prompt_len = inputs.shape[1]

outputs = llm.model.generate(inputs, max_new_tokens=512, do_sample=True, top_k=50, top_p=0.95, num_return_sequences=3)
for i, output in enumerate(outputs):
  print('Summary', i)
  summary = llm.tokenizer.decode(output[prompt_len+1:], skip_special_tokens=True)
  print(summary)
  print()


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Summary 0
 following are excerpts of her Facebook post:
So, my brother's position on same-sex marriage and the 'dangers of same-sex marriage' is that it should be determined by voters in the states, not by anyone in the Legislature or judiciary.  As you have already noted, I support same-sex marriage and am glad that people are watching this.  I am going to challenge Senator Enzi if he is elected and I intend to take his seat.  I was once my favorite Democrat in office—I don't know what it is going to take but I'll be watching the debates from here on out. 
So, I am excited to start this campaign and look forward to it being my first fight in the House.  So, I am hoping to win with a strong majority (not a margin of victory)  and I am excited that I will be able to get an even bigger platform than I do in the Senate because I know I want to do the right thing.  The problem is not just about the Democrats—it's also about the Republicans, which are the major party in Washington, DC and t

In [125]:
async def gen_summary(llm_name, llm, bias_model, steer_model, article, target_bias):

  prompt = f'Summarize this article: {article}'

  if llm_name in ['gpt2']:
    print('Note: llm is gpt2 so prepend endoftext')
    prompt = f'<|endoftext|>{prompt}'

  model = steer_model(llm, bias_model, prompt, target_bias)

  particles = await smc_standard(model, 10)

  for i, p in enumerate(particles):
    print(f'Summary {i+1}:')

    summary = str(p.context.s)[p.prompt_len+1:-1]
    print(summary)

    pred_bias, _ = bias_model(summary)
    print(pred_bias)
    print(p.weight)
    print()

  weights = np.array([p.weight for p in particles])
  if weights.shape[0] == 0:
    print('No successful particles!')
    return ''

  best_particle = particles[weights.argmax()]
  return str(best_particle.context.s)[p.prompt_len+1:-1]

Generate using Naive Model

In [130]:
summary = await gen_summary(model_name, llm, bias_model, NaiveModel, article, 'center')
print(summary)

Note: llm is gpt2 so prepend endoftext
next sentence: 
Elizabeth Cheney posted:
left


CancelledError: ignored

ERROR:asyncio:Exception in callback CachedCausalLM.add_query.<locals>.<lambda>() at /usr/local/lib/python3.10/dist-packages/hfppl/llms.py:321
handle: <TimerHandle when=9918.247566044 CachedCausalLM.add_query.<locals>.<lambda>() at /usr/local/lib/python3.10/dist-packages/hfppl/llms.py:321>
Traceback (most recent call last):
  File "/usr/lib/python3.10/asyncio/events.py", line 80, in _run
    self._context.run(self._callback, *self._args)
  File "/usr/local/lib/python3.10/dist-packages/hfppl/llms.py", line 321, in <lambda>
    self.timer = asyncio.get_running_loop().call_later(self.timeout, lambda: self.batch_evaluate_queries())
  File "/usr/local/lib/python3.10/dist-packages/torch/utils/_contextlib.py", line 115, in decorate_context
    return func(*args, **kwargs)
  File "/usr/local/lib/python3.10/dist-packages/hfppl/llms.py", line 309, in batch_evaluate_queries
    q.future.set_result(results.logits[i])
asyncio.exceptions.InvalidStateError: invalid state


Generate using TwistModel

In [ ]:
summary = await gen_summary(model_name, llm, bias_model, TwistModel, article, 'center')
print(summary)

Note: llm is gpt2 so prepend endoftext
next sentence: 
-2.0077918
next sentence: 
North Dakota that much is clear in both cases, State Rep.
-0.41448626
next sentence: 
In a public statement, Larry Kasowitz, press secretary for Sen.
-2.4334092
next sentence:  The newspaper says her public response was a response to the Texan's response in 2014.
-1.7958852
next sentence: 
Citing Shakespeare, the 1980 novel, the sociology professor suggests that the present day public opinion evolves around the topic of such matters as single parents and gay marriage.
-1.4350029
next sentence:  In May, she wrote of same-sex marriage on her Facebook page, on the same reasoning for which she has also referenced same-sex relationships, suggesting marriage is equally valid in its own right.
1.6466471
next sentence: 
Regardless of whether the tit-for-tat controversy surrounding Representative Mike Enzi’s current race is an aberration or commonplace, perhaps of little concern to voters in 2016 in this country w

# Misc

In [ ]:
val_df = df.sample(n=200).reset_index(drop=True)
val_df.head()

In [ ]:
preds = []
for body in val_df['body']:
  preds.append(bias_model(body))

In [ ]:
val_df['pred'] = preds

In [ ]:
val_df.head()

In [ ]:
correct_preds = val_df[val_df['stance'] == val_df['pred']]
print(correct_preds.shape[0])
print(correct_preds.shape[0]/200)
correct_preds.head()

In [ ]:
incorrect_preds = val_df[val_df['stance'] != val_df['pred']]
print(incorrect_preds.shape[0])
incorrect_preds.head()